# ВААК: выравнивание русских и абазинских лонгридов

Ноутбук собирает параллельный корпус: пары предложений **русский / абазинский**.

## Как устроено выравнивание

Изначально 111 пар ссылок: одна ведёт на материал на абазинском, вторая — на тот же материал на русском.

Мы **не** сравниваем абазинский с русским напрямую. Абазинский малоресурсный, модель плохо его «видит». Поэтому:

1. Скачиваем оба текста со страниц.
2. Режем на предложения (Lingtrain).
3. Каждое абазинское предложение переводим на русский через Яндекс.Translate. Это только **мост**: в итоговый корпус машинный перевод не попадает.
4. Модель **LaBSE** выравнивает два русских текста: оригинал с сайта и машинный перевод с абазинского (семантическая близость).
5. Найденному русскому переводу подставляем обратно **исходное** абазинское предложение.
6. Результат — таблица пар `russian` / `abaza` + ссылка и заголовок, файл xlsx.

## Как запускать

Ячейки — **по порядку, по одной**.

### Что за Яндекс

Чтобы перевести абазинские предложения на русский, нужен **Yandex Cloud Translate** (не «переводчик на сайте», а API в облаке). В Colab кладутся два секрета:

- `YANDEX_API_KEY` — ключ доступа к API (длинная строка, которую выдаёт консоль Яндекса после «Создать API-ключ»).
- `YANDEX_FOLDER_ID` — идентификатор **каталога** в облаке, тот, что начинается с `b1g…`. Не путать с id сервисного аккаунта (`aje…`) — тот сюда не подходит.

Без этих двух значений ячейка перевода упадёт. Сами ключи в код вписывать не нужно: ноутбук берёт их из секретов Colab (или спросит в поле ввода).

### Какой Excel подавать

Можно **подставить другой файл** со ссылками. В ячейке загрузки просто выбрать свой xlsx (или tsv) вместо `vaak_sheet4_links.xlsx`.

Нужна таблица, **одна строка = одна статья**, две ссылки на один и тот же материал:

| что | как назвать колонку | пример |
| --- | --- | --- |
| ссылка на абазинский текст | в имени есть `url` и `abq` (как `url_abq`) | `https://…/abq/…` |
| ссылка на русский текст | в имени есть `url` и `ru` или `guess` (как `url_ru` / `url_ru_guess`) | `https://…` без `/abq/` |
| заголовок (необязательно) | колонка с `title` в имени | название статьи |

Код **сам ищет колонки по этим кускам имени**, жёстко прописанных `url_abq` / `url_ru_guess` в логике нет. Если русскую колонку не найдёт — попробует собрать русскую ссылку из абазинской (уберёт `/abq/` или `/abk/` из пути). Это сработает только если у сайта так устроены URL.

**Менять код не нужно**, если колонки названы похоже и это пары страниц того же сайта (ВААК). Менять ячейку с таблицей придётся, только если колонки называются совсем иначе (`link1`, `abaza_page` и т.п.) — тогда либо переименовать их в Excel, либо поправить поиск колонок в ячейке 4.

После скачивания в логе должны быть `ok` и длинные тексты в `ru`/`abq`. Потом ячейка Lingtrain. Готовый файл: Google Drive → `vaak_align/vaak_aligned.xlsx`.


In [ ]:
# Ячейка 1. Библиотеки для скачивания страниц
# Playwright открывает сайт как браузер (нужен Chromium).
# lingtrain-aligner и sentence-transformers понадобятся позже, на шаге выравнивания.

!pip -q install lingtrain-aligner requests openpyxl pandas sentence-transformers playwright nest_asyncio
!playwright install chromium
!playwright install-deps chromium


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 MB 23.0 MB/s eta 0:00:00
186.8 MiB [] 0% 0.0s186.8 MiB [] 0% 23.3s186.8 MiB [] 0% 12.0s186.8 MiB [] 0% 8.4s186.8 MiB [] 0% 7.0s186.8 MiB [] 1% 5.3s186.8 MiB [] 2% 3.8s186.8 MiB [] 3% 3.1s186.8 MiB [] 4% 2.8s186.8 MiB [] 5% 2.5s186.8 MiB [] 6% 2.3s186.8 MiB [] 6% 2.5s186.8 MiB [] 7% 2.7s186.8 MiB [] 7% 2.8s186.8 MiB [] 7% 2.9s186.8 MiB [] 8% 2.7s186.8 MiB [] 9% 2.5s186.8 MiB [] 10% 2.3s186.8 MiB [] 12% 2.2s186.8 MiB [] 13% 2.1s186.8 MiB [] 14% 2.0s186.8 MiB [] 15% 2.0s186.8 MiB [] 16% 1.9s186.8 MiB [] 17% 1.8s186.8 MiB [] 18% 1.8s186.8 MiB [] 19% 1.8s186.8 MiB [] 20% 1.9s186.8 MiB [] 20% 1.8s186.8 MiB [] 22% 1.8s186.8 MiB [] 23% 1.7s186.8 MiB [] 24% 1.7s186.8 MiB [] 24% 1.8s186.8 MiB [] 25% 1.7s186.8 MiB [] 27% 1.6s186.8 MiB [] 28% 1.6s186.8 MiB [] 30% 1.5s186.8 MiB [] 31% 1.4s186.8 MiB [] 33% 1.4s186.8 MiB [] 34% 1.3s186.8 MiB [] 35% 1.3s186.8 MiB [] 36% 1

In [ ]:
# Ячейка 2. Остальные зависимости Lingtrain
# razdel — нарезка русского на предложения; sklearn/scipy — для выравнивания.
# Если что-то уже стоит из ячейки 1, pip просто пропустит.

!pip -q install \
  lingtrain-aligner \
  dateparser python-dateutil razdel \
  sentence-transformers \
  tqdm numpy scipy scikit-learn \
  matplotlib seaborn lxml \
  openpyxl pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 kB 14.3 MB/s eta 0:00:00


In [ ]:
# Ячейка 3. Ключи Яндекса и таблица ссылок
# Читаем API-ключ и Folder ID (каталог default, начинается с b1g…).
# Затем загружаем Excel: 111 пар URL (абазинская страница + русская версия того же материала).

from google.colab import files, userdata
from getpass import getpass

def secret(name, prompt):
    try:
        v = userdata.get(name)
        if v:
            return v.strip()
    except Exception:
        pass
    return getpass(prompt).strip()

YANDEX_API_KEY = secret("YANDEX_API_KEY", "YANDEX_API_KEY: ")
YANDEX_FOLDER_ID = secret("YANDEX_FOLDER_ID", "YANDEX_FOLDER_ID (b1g… каталог default): ")
assert YANDEX_FOLDER_ID.startswith("b1"), (
    "Это не Folder ID. Нужен идентификатор каталога default: b1ggvg80f2nuvbgitudq. "
    f"Сейчас: {YANDEX_FOLDER_ID[:12]!r}"
)
print("ключи ок, folder", YANDEX_FOLDER_ID)

print("Загрузите vaak_sheet4_links.xlsx")
up = files.upload()
LINKS_PATH = next(iter(up))


ключи ок, folder b1ggvg80f2ruvbgitudq
Загрузите vaak_sheet4_links.xlsx


Saving vaak_sheet4_links (1).xlsx to vaak_sheet4_links (1).xlsx


In [ ]:
# Ячейка 4. Разбираем таблицу ссылок
# Ищем колонки с абазинским и русским URL, нормализуем имена.
# Если русской ссылки нет — пробуем получить её из абазинской (убрать /abq/ из пути).

import os, re, time, requests, pandas as pd

df = pd.read_excel(LINKS_PATH) if str(LINKS_PATH).endswith((".xlsx", ".xls")) else pd.read_csv(LINKS_PATH, sep="\t")
df.columns = [c.strip() for c in df.columns]
col_abq = [c for c in df.columns if "abq" in c.lower() and "url" in c.lower()]
col_ru = [c for c in df.columns if "url" in c.lower() and ("ru" in c.lower() or "guess" in c.lower())]
col_title = [c for c in df.columns if "title" in c.lower()]
assert col_abq, list(df.columns)
df["url_abq"] = df[col_abq[0]].astype(str)
if col_ru:
    df["url_ru"] = df[col_ru[0]].astype(str)
else:
    df["url_ru"] = df["url_abq"].str.replace("/abq/", "/", regex=False).str.replace("/abk/", "/", regex=False)
df["title"] = df[col_title[0]] if col_title else df["url_abq"]
print(len(df), "строк")
df[["title", "url_abq", "url_ru"]].head(3)


111 строк


,title,url_abq,url_ru
0,Амамыри ауасаражври ргIарбага,https://abaza.org/abq/amamyri-auasarazhuri-rga...,https://abaza.org/amamyri-auasarazhuri-rgarbaga
1,Келешбейи апсыуаква ргвали,https://abaza.org/abq/keleshbeyi-apsyuakua-rguali,https://abaza.org/keleshbeyi-apsyuakua-rguali
2,Абзазара ахъаз аджьаль апшIалра,https://abaza.org/abq/abzazara-aqhaz-adzhiali-...,https://abaza.org/abzazara-aqhaz-adzhiali-apts...


In [ ]:
# Ячейка 5. Функция перевода Яндекса (только мост, не корпус)
# Переводит пачки абазинских предложений на русский.
# Этот текст нужен, чтобы LaBSE могла сравнить «русский с сайта» и «русский перевод абазинского».
# В итоговый xlsx эти переводы не записываются.

def yandex_translate(texts, source, target="ru", sleep=0.15):
    url = "https://translate.api.cloud.yandex.net/translate/v2/translate"
    headers = {"Authorization": f"Api-Key {YANDEX_API_KEY}"}
    out, i = [], 0
    while i < len(texts):
        chunk, size = [], 0
        while i < len(texts) and size + len(texts[i]) < 9000 and len(chunk) < 80:
            chunk.append(texts[i])
            size += len(texts[i]) + 1
            i += 1
        body = {
            "folderId": YANDEX_FOLDER_ID,
            "texts": chunk,
            "sourceLanguageCode": source,
            "targetLanguageCode": target,
        }
        resp = requests.post(url, json=body, headers=headers, timeout=90)
        if resp.status_code >= 400:
            raise RuntimeError(resp.status_code, resp.text[:800])
        out.extend(t.get("text", "") for t in resp.json()["translations"])
        time.sleep(sleep)
    return out

print("yandex_translate готов")


yandex_translate готов


In [ ]:
# Ячейка 6. Скачивание текстов с сайта
# Для каждой пары ссылок открываем обе страницы в Chromium и забираем текст.
# Грубая чистка: выкидываем служебные строки (cookie, соцсети, шапка конгресса) — это ещё не выравнивание.
# Короткие/пустые страницы пропускаем. Удачные пары пишем на диск: Drive/vaak_align/rows_ok.jsonl
# Можно перезапускать: уже скачанные URL не качаем заново.

from google.colab import drive
drive.mount("/content/drive")

import os, json, asyncio, nest_asyncio, re
from pathlib import Path
from playwright.async_api import async_playwright

nest_asyncio.apply()

SAVE_DIR = Path("/content/drive/MyDrive/vaak_align")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
CKPT = SAVE_DIR / "rows_ok.jsonl"
ERRF = SAVE_DIR / "errors.jsonl"

DROP = re.compile(
    r"(cookie|подпис|telegram|instagram|facebook|всемирный абхазо-абазинский конгресс|адуней апсуа-абаза конгресс)",
    re.I,
)

def _clean(raw: str) -> str:
    lines = [ln.strip() for ln in raw.splitlines() if len(ln.strip()) >= 2 and not DROP.search(ln)]
    return "\n".join(lines)

def load_done():
    done = set()
    if CKPT.exists():
        with CKPT.open(encoding="utf-8") as f:
            for line in f:
                rec = json.loads(line)
                done.add(rec["url_abq"])
    return done

def append(path, rec):
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

async def fetch_one(page, url: str) -> str:
    await page.goto(url, wait_until="domcontentloaded", timeout=90000)
    try:
        await page.wait_for_function("document.body.innerText.length > 80", timeout=12000)
    except Exception:
        pass
    await page.wait_for_timeout(1500)
    return _clean(await page.inner_text("body"))

async def run():
    N_LIMIT = None
    SKIP_SHORT = 400
    work = df.head(N_LIMIT) if N_LIMIT else df.copy()
    done = load_done()
    print("уже на диске", len(done))
    errors = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=["--no-sandbox", "--disable-dev-shm-usage"],
        )
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
        )
        page = await context.new_page()

        n_ok = len(done)
        for i, r in work.iterrows():
            url_abq, url_ru = str(r["url_abq"]), str(r["url_ru"])
            title = str(r.get("title", ""))
            if url_abq in done:
                print("уже есть", i, title[:40])
                continue
            try:
                ru = await fetch_one(page, url_ru)
                abq = await fetch_one(page, url_abq)
                print(i, "ru", len(ru), "abq", len(abq), title[:50])
                if min(len(ru), len(abq)) < SKIP_SHORT:
                    rec = {"title": title, "url_abq": url_abq, "error": f"пусто ru={len(ru)} abq={len(abq)}"}
                    errors.append(rec)
                    append(ERRF, rec)
                    print("skip пустая")
                    continue
                rec = {
                    "n": int(i), "title": title, "url_abq": url_abq, "url_ru": url_ru,
                    "ru": ru, "abq": abq,
                }
                append(CKPT, rec)
                done.add(url_abq)
                n_ok += 1
                print("ok", n_ok, "saved")
            except Exception as e:
                rec = {"title": title, "url_abq": url_abq, "error": str(e)[:500]}
                errors.append(rec)
                append(ERRF, rec)
                print("ERR", title[:40], type(e).__name__, e)

        await context.close()
        await browser.close()

    rows_ok = []
    if CKPT.exists():
        rows_ok = [json.loads(l) for l in CKPT.read_text(encoding="utf-8").splitlines() if l.strip()]
    return rows_ok, errors

rows_ok, errors = asyncio.get_event_loop().run_until_complete(run())
print("итого скачано", len(rows_ok), "новых ошибок в этом запуске", len(errors))
print("файлы:", CKPT, ERRF)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
уже на диске 0
0 ru 9306 abq 3 Амамыри ауасаражври ргIарбага
skip пустая
1 ru 11680 abq 11954 Келешбейи апсыуаква ргвали
ok 1 saved
2 ru 26060 abq 28471 Абзазара ахъаз аджьаль апшIалра
ok 2 saved
3 ru 15034 abq 15662 йахъыцIуа асквш 100 йадахIвуата: йапхъахауа апсыуа
ok 3 saved
4 ru 9452 abq 10551 Ахъахвитра йащатахаз ацзакIыра: асквш 1989 агIан Л
ok 4 saved
5 ru 9172 abq 8843 Адуней йырзыхътIу анапIсаргвыцIа: апсуакви абазакв
ok 5 saved
6 ru 16173 abq 16990 Апсуа тшчвашвкIыгIв: «апхъанчIви айсчва ркърар» нк
ok 6 saved
7 ru 12492 abq 13572 https://abaza.org/abq/aysra-asuratkua-zguy-yyztshn
ok 7 saved
8 ru 8799 abq 10230 КIавкIаз ауагIахъаква рконфедерация ангIанша амш й
ok 8 saved
9 ru 9542 abq 9973 Ауат йрызпшитI: август а 30 Апсны апны хабардъа йы
ok 9 saved
10 ru 10774 abq 11252 Апсны абльын Ду: азцIгIаракви аджьауапква рзыпшгIа
ok 10 saved
11 ru 4681 abq 

In [ ]:
# Ячейка 7. Нарезка, перевод-мост, выравнивание LaBSE, сбор пар
# 1) Оба текста режем на предложения (Lingtrain splitter).
# 2) Абазинские предложения переводим на русский через Яндекс (proxy_to).
# 3) В базу кладём: русский оригинал и абазинский оригинал, но для сравнения модели
#    подсовываем русский перевод абазинского (use_proxy_to=True).
# 4) LaBSE ищет семантически близкие пары двух русских текстов (окно window=40).
# 5) resolver закрывает разрывы (1-ко-многим, пропуски).
# 6) В пару записываем русский оригинал + исходный абазинский (не машинный перевод).
# Результат по статьям в aligned/*.json и общий Excel: vaak_align/vaak_aligned.xlsx

from google.colab import drive
drive.mount("/content/drive")

from lingtrain_aligner import preprocessor, splitter, aligner, resolver, helper
import os, json
from pathlib import Path
import pandas as pd

SAVE_DIR = Path("/content/drive/MyDrive/vaak_align")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
CKPT_SCRAPE = SAVE_DIR / "rows_ok.jsonl"
ALIGN_DIR = SAVE_DIR / "aligned"
ALIGN_DIR.mkdir(exist_ok=True)
ALIGN_ERR = SAVE_DIR / "align_errors.jsonl"
XLSX_DRIVE = SAVE_DIR / "vaak_aligned.xlsx"
DB_DIR = Path("align_db")
DB_DIR.mkdir(exist_ok=True)

ABQ_CODE = "abq"
MODEL = "sentence_transformer_multilingual_labse"
SKIP_SHORT = 400

def load_jsonl(path):
    if not path.exists():
        return []
    with path.open(encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

def append_err(rec):
    with ALIGN_ERR.open("a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

rows_ok = load_jsonl(CKPT_SCRAPE)
assert rows_ok, f"нет {CKPT_SCRAPE} — сначала докачайте Playwright"

print("текстов на диске", len(rows_ok), "уже выровнено", len(list(ALIGN_DIR.glob("*.json"))))

aligned_rows, align_errors = [], []
for k, item in enumerate(rows_ok):
    url = item["url_abq"]
    slug = f"{int(item.get('n', k)):03d}"
    outp = ALIGN_DIR / f"{slug}.json"
    if outp.exists():
        pairs = json.loads(outp.read_text(encoding="utf-8"))
        aligned_rows.extend(pairs)
        print("skip уже есть", slug, item["title"][:40], "pairs", len(pairs))
        continue
    if min(len(item.get("ru") or ""), len(item.get("abq") or "")) < SKIP_SHORT:
        rec = {"url": url, "title": item["title"], "error": "пусто"}
        align_errors.append(rec)
        append_err(rec)
        print("skip пустая", slug)
        continue

    ru_lines = preprocessor.mark_paragraphs(item["ru"].splitlines())
    abq_lines = preprocessor.mark_paragraphs(item["abq"].splitlines())
    spl_ru = splitter.split_by_sentences_wrapper(ru_lines, "ru")
    spl_abq = splitter.split_by_sentences_wrapper(abq_lines, "ru")
    plain_abq = [x[0] if isinstance(x, (list, tuple)) else x for x in spl_abq]
    try:
        proxy = yandex_translate(plain_abq, source=ABQ_CODE, target="ru")
    except RuntimeError as e:
        rec = {"url": url, "title": item["title"], "error": f"yandex {e}"}
        align_errors.append(rec)
        append_err(rec)
        print("YANDEX FAIL", item["title"][:40], e)
        continue

    db = str(DB_DIR / f"{slug}.db")
    if os.path.exists(db):
        os.remove(db)
    aligner.fill_db(
        db, lang_from="ru", lang_to="ru",
        splitted_from=spl_ru, splitted_to=spl_abq,
        proxy_to=proxy, name=item["title"][:80],
    )
    n_from = len(spl_ru)
    batch_ids = list(range(max(1, (n_from + 99) // 100)))
    aligner.align_db(
        db, MODEL, batch_size=100, window=40, batch_ids=batch_ids,
        save_pic=False, embed_batch_size=32, normalize_embeddings=True,
        show_progress_bar=True, use_proxy_to=True,
    )
    try:
        resolver.resolve_all(db)
    except Exception:
        try:
            resolver.auto_resolve(db)
        except Exception as e:
            print("resolve skip", e)

    doc_from, doc_to = helper.read_processing(db)
    pairs = []
    for ru_s, abq_s in zip(doc_from, doc_to):
        ru_s, abq_s = (ru_s or "").strip(), (abq_s or "").strip()
        if not ru_s and not abq_s:
            continue
        pairs.append({
            "russian": ru_s, "abaza": abq_s,
            "url": url, "url_ru": item["url_ru"], "title": item["title"],
        })
    outp.write_text(json.dumps(pairs, ensure_ascii=False), encoding="utf-8")
    aligned_rows.extend(pairs)
    print("aligned", slug, item["title"][:50], "pairs", len(pairs))
    pd.DataFrame(aligned_rows).to_excel(XLSX_DRIVE, index=False)

print("всего пар", len(aligned_rows), "новых ошибок выравнивания", len(align_errors))
print("Excel на диске", XLSX_DRIVE)


Mounted at /content/drive
текстов на диске 109 уже выровнено 67
skip уже есть 001 Келешбейи апсыуаква ргвали pairs 81
skip уже есть 002 Абзазара ахъаз аджьаль апшIалра pairs 173
skip уже есть 003 йахъыцIуа асквш 100 йадахIвуата: йапхъах pairs 118
skip уже есть 004 Ахъахвитра йащатахаз ацзакIыра: асквш 19 pairs 60
skip уже есть 005 Адуней йырзыхътIу анапIсаргвыцIа: апсуак pairs 64
skip уже есть 006 Апсуа тшчвашвкIыгIв: «апхъанчIви айсчва  pairs 86
skip уже есть 007 https://abaza.org/abq/aysra-asuratkua-zg pairs 141
skip уже есть 008 КIавкIаз ауагIахъаква рконфедерация ангI pairs 56
skip уже есть 009 Ауат йрызпшитI: август а 30 Апсны апны х pairs 71
skip уже есть 010 Апсны абльын Ду: азцIгIаракви аджьауапкв pairs 68
skip уже есть 011 Апсны апны асквш асуратчпагIв дгIалырхтI pairs 42
skip уже есть 012 Адиннкъвгарта йгьи архитектура ачвахъаба pairs 70
skip уже есть 013 «Абазашта» газет: абаза уагIахъа рыбызшв pairs 52
skip уже есть 014 Апхъазаман адамыгъа хIагIа: Согъвым ауна pairs 89
skip

modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/2.02k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

resolve skip module 'lingtrain_aligner.resolver' has no attribute 'auto_resolve'
aligned 109 Абазаква ртурых pairs 34
Aligning without segments.
tasks amount: 1
batch: 0 (0)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

resolve skip module 'lingtrain_aligner.resolver' has no attribute 'auto_resolve'
aligned 110 Къарча-Черкес апсабара pairs 14
всего пар 10461 новых ошибок выравнивания 2
Excel на диске /content/drive/MyDrive/vaak_align/vaak_aligned.xlsx
